# 장바구니에 담은 상품의 7일 내 구매 여부 분석

화장품 온라인 스토어의 2019년 10~11월 행동 로그를 고객 단위로 재구성하고, 담는 시점까지 알 수 있는 정보로 같은 상품의 7일 내 구매 여부를 분류한 프로젝트다.

- 원본: REES46 Marketing Platform, *E-commerce Events History in Cosmetics Shop*
- 관측 단위: 고객별 최초 장바구니 담기 1건
- 목표변수: `purchase_7d`
- 최종 모형: 해석 중심 로지스틱 회귀

원본 CSV와 고객별 예측값은 공개하지 않는다. 앞부분은 원본이 있을 때 실행할 수 있는 처리 구조이며, 뒷부분은 최종 산출물과 대조한 집계 결과다.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

RAW_FILES = [Path("data/2019-Oct.csv"), Path("data/2019-Nov.csv")]
CHUNK_SIZE = 1_000_000
WINDOW = pd.Timedelta(minutes=30)
TARGET_HORIZON = pd.Timedelta(days=7)
BURN_IN_START = pd.Timestamp("2019-10-11 00:00:00")

## 1. 원본 데이터와 품질 점검

한 행은 고객 한 명이 아니라 조회, 장바구니 담기, 담기 취소 또는 구매 행동 하나다. 두 월별 파일은 열 구성이 같아 세로로 결합한다. 대용량 파일은 100만 행씩 나눠 읽고 불러온 행 수를 파일 줄 수와 대조했다.

In [ ]:
USECOLS = [
    "event_time", "event_type", "product_id", "category_id",
    "category_code", "brand", "price", "user_id", "user_session",
]


def read_events(paths=RAW_FILES, chunksize=CHUNK_SIZE):
    frames = []
    for path in paths:
        for chunk in pd.read_csv(path, usecols=USECOLS, chunksize=chunksize):
            frames.append(chunk)
    return pd.concat(frames, ignore_index=True)


def clean_events(events):
    out = events.copy()
    utc = pd.to_datetime(out["event_time"], utc=True, errors="raise")
    out["event_time"] = (utc + pd.Timedelta(hours=3)).dt.tz_localize(None)
    out = out.drop_duplicates(
        subset=["user_id", "product_id", "event_type", "event_time"]
    )
    out = out.loc[out["price"].gt(0)].copy()
    return out.sort_values(["user_id", "event_time"])


# raw = read_events()
# events = clean_events(raw)
# assert len(raw) == 8_738_120
# assert len(events) == 8_259_159

품질 점검에서 고객·상품·행동·시각이 같은 461,449행과 가격이 0 이하인 17,512행을 제거했다. 시간대는 활동 분포를 근거로 UTC+3을 적용했으며 제공처 문서로 확인한 값이 아니므로 한계로 남겼다.

## 2. 고객 단위 관측치와 목표변수

고객별 최초 담기만 남기고, 7일을 온전히 관찰할 수 없는 종료 구간과 기록 시작 전 행동이 잘린 초기 구간을 제외한다. 같은 초에 여러 상품을 처음 담은 고객도 기준 상품을 정할 근거가 없어 제외한다.

In [ ]:
def select_first_cart_cohort(events, burn_in_start=BURN_IN_START):
    carts = events.loc[events["event_type"].eq("cart")].copy()
    first_time = carts.groupby("user_id")["event_time"].transform("min")
    first = carts.loc[carts["event_time"].eq(first_time)].copy()

    products_at_first_time = first.groupby("user_id")["product_id"].transform("nunique")
    first = first.loc[products_at_first_time.eq(1)]
    first = first.drop_duplicates("user_id").rename(columns={"event_time": "t0"})

    observation_cutoff = events["event_time"].max() - TARGET_HORIZON
    keep = first["t0"].ge(burn_in_start) & first["t0"].le(observation_cutoff)
    return first.loc[keep, ["user_id", "t0", "product_id", "price", "brand"]]


def attach_purchase_target(cohort, events):
    base = cohort.reset_index(drop=True).reset_index(names="row_id")
    purchases = events.loc[
        events["event_type"].eq("purchase"),
        ["user_id", "product_id", "event_time"],
    ]
    pairs = base.merge(purchases, on=["user_id", "product_id"], how="left")
    valid = pairs["event_time"].gt(pairs["t0"]) & pairs["event_time"].le(
        pairs["t0"] + TARGET_HORIZON
    )
    first_purchase = pairs.loc[valid].groupby("row_id")["event_time"].min()

    base["purchase_7d"] = base["row_id"].isin(first_purchase.index).astype("int8")
    base["lag_hours"] = (
        base["row_id"].map(first_purchase).sub(base["t0"]).dt.total_seconds() / 3600
    )
    return base.drop(columns="row_id")

## 3. 담기 이전 30분 행동 변수

설명변수는 담는 순간까지 알 수 있는 값으로 제한한다. `remove_from_cart`, 구매 시각과 구매까지 걸린 시간은 담은 뒤에 알 수 있어 설명변수에 넣지 않는다.

In [ ]:
def attach_pre_cart_features(cohort, events):
    base = cohort.reset_index(drop=True).reset_index(names="row_id")
    views = events.loc[
        events["event_type"].eq("view"),
        ["user_id", "product_id", "event_time"],
    ].rename(columns={"product_id": "viewed_product_id"})

    pairs = base[["row_id", "user_id", "t0", "product_id"]].merge(
        views, on="user_id", how="left"
    )
    within_window = pairs["event_time"].ge(pairs["t0"] - WINDOW) & pairs[
        "event_time"
    ].lt(pairs["t0"])
    recent = pairs.loc[within_window].copy()
    recent["same_product"] = recent["viewed_product_id"].eq(recent["product_id"])

    features = recent.groupby("row_id").agg(
        prior_view_cnt=("viewed_product_id", "size"),
        prior_uniq_product_cnt=("viewed_product_id", "nunique"),
        cart_product_view_cnt=("same_product", "sum"),
    )
    base = base.join(features, on="row_id")
    count_cols = [
        "prior_view_cnt", "prior_uniq_product_cnt", "cart_product_view_cnt"
    ]
    base[count_cols] = base[count_cols].fillna(0).astype("int32")
    base["has_prior_view"] = base["prior_view_cnt"].gt(0).astype("int8")
    base["cart_timeband"] = pd.cut(
        base["t0"].dt.hour,
        bins=[-1, 6, 12, 18, 24],
        labels=["심야", "오전", "오후", "저녁"],
        right=False,
    )
    base["cart_weekday"] = base["t0"].dt.day_name()
    return base.drop(columns="row_id")

실제 대용량 파이프라인에서는 고객·시각 정렬과 중간 산출물 저장으로 메모리 사용을 나눴다. 위 함수는 관측 창과 집계 정의를 한곳에서 확인하기 위한 공개용 구조다.

## 4. 최종 표본과 변수

In [ ]:
cohort_summary = pd.DataFrame(
    {
        "단계": [
            "장바구니에 담은 고객", "관측 종료 직전 제외",
            "관측 초기 구간 제외", "동시각 복수 상품 제외",
        ],
        "남은 고객": [210_593, 190_925, 108_630, 108_458],
        "제외 고객": [0, 19_668, 82_295, 172],
    }
)

target_summary = pd.Series(
    {"전체": 108_458, "7일 내 구매": 22_236, "구매 비율": 22_236 / 108_458}
)

assert target_summary["전체"] == 108_458
assert np.isclose(target_summary["구매 비율"], 0.205019, atol=1e-6)
cohort_summary, target_summary

In [ ]:
final_features = pd.DataFrame(
    [
        ("price", "연속형", "log"),
        ("prior_view_cnt", "연속형", "log1p 후 중심화·제곱항"),
        ("cart_product_view_cnt", "연속형", "log1p 후 중심화·제곱항"),
        ("brand_grp", "명목형", "학습용 기준으로 더미 인코딩"),
        ("cart_timeband", "명목형", "학습용 기준으로 더미 인코딩"),
        ("cart_weekday", "명목형", "학습용 기준으로 더미 인코딩"),
    ],
    columns=["변수", "척도", "전처리"],
)
final_features

학습용 86,766명과 검증용 21,692명으로 층화 분할한 뒤 전처리 기준은 학습용에서만 계산했다. 로짓 선형성 위배가 순차적으로 확인된 두 조회 변수에 중심화 제곱항을 추가했다. 범주 더미를 포함한 최종 모형은 28개 항이다.

## 5. 모형 성능과 비교

In [ ]:
performance = pd.DataFrame(
    {
        "구분": ["학습", "5겹 교차검증", "검증"],
        "ROC_AUC": [0.591513, 0.589969, 0.584078],
        "표준편차": [np.nan, 0.005143, np.nan],
    }
)

test_metrics_at_019 = pd.Series(
    {
        "threshold": 0.19,
        "precision": 0.239932,
        "recall": 0.667191,
        "F1": 0.352941,
        "PR_AUC": 0.2609,
    }
)

assert np.isclose(performance.loc[2, "ROC_AUC"], 0.584078)
performance, test_metrics_at_019

In [ ]:
model_comparison = pd.DataFrame(
    [
        ("로지스틱 회귀", 0.5841, 0.0000),
        ("의사결정나무", 0.5888, 0.0047),
        ("랜덤포레스트", 0.5922, 0.0081),
        ("HistGradientBoosting", 0.5964, 0.0123),
        ("XGBoost", 0.5954, 0.0114),
        ("LightGBM", 0.5969, 0.0128),
        ("CatBoost", 0.5911, 0.0071),
    ],
    columns=["모형", "검증 ROC_AUC", "로지스틱 대비 차이"],
)
model_comparison.sort_values("검증 ROC_AUC", ascending=False)

최고 성능은 LightGBM의 0.5969였지만 로지스틱 회귀보다 0.0128 높았다. 사전에 정한 0.02 기준보다 작아 계수와 오즈비로 방향을 설명할 수 있는 로지스틱 회귀를 최종 모형으로 유지했다. 이 기준은 보편적 규칙이 아니라 프로젝트 시작 전에 정한 운영 규칙이다.

## 6. 변수 영향력과 리프트

In [ ]:
top_effects = pd.DataFrame(
    [
        ("담은 상품 조회 횟수", 0.279678, 2.054961),
        ("가격(log)", 0.152135, 1.144246),
        ("시간대: 오후", 0.110208, 1.258108),
        ("전체 조회 횟수", -0.109134, 0.841553),
    ],
    columns=["조건", "표준화 계수", "오즈비"],
)

lift_summary = pd.DataFrame(
    [
        ("상위 5%", 1_084, 0.3146, 1.53),
        ("상위 10%", 2_170, 0.2986, 1.46),
        ("상위 20%", 4_338, 0.2842, 1.39),
        ("상위 30%", 6_507, 0.2643, 1.29),
    ],
    columns=["선별 범위", "고객", "실제 구매 비율", "누적 리프트"],
)

assert np.isclose(lift_summary.loc[1, "누적 리프트"], 1.46)
top_effects, lift_summary

담은 상품의 직전 조회 횟수가 가장 큰 신호였다. 같은 상품을 2회 이상 본 고객 6,071명의 구매 비율은 28.84%였다. 다만 검증용 예측확률 상위 10%에서도 70.1%가 구매하지 않아 개별 고객 판정이나 비용이 드는 타기팅에는 사용할 수 없다.

## 7. 해석 범위와 한계

- 구매자의 63.57%가 담은 뒤 1시간 안에 구매했다. 담기 이전 정보만으로 이후 행동을 구분할 여지가 좁다.
- 7일 이후 같은 상품을 구매한 2,267건은 목표변수에서 미구매로 기록된다.
- 미구매 86,222명 가운데 6,460명은 7일 안에 다른 상품을 구매했다.
- 시간대와 관측 초기 제외 기준은 행동 분포를 근거로 정한 판단이다.
- 상품 카테고리 통제가 제한되어 가격 계수에 교란이 남을 수 있다.
- 한 스토어의 2019년 두 달 기록이므로 다른 업종과 시점에 바로 일반화할 수 없다.

분석 결과는 개별 고객의 구매를 맞히는 모델보다 장바구니 화면에서 어떤 고객 조건을 먼저 살펴볼지 정하는 저비용 보조 신호로 해석한다. 실제 개입 효과는 A/B 테스트로 확인해야 한다.